In [ ]:
import pandas as pd
import numpy as np
import os
from xgboost import XGBRegressor
import time
from sklearn.model_selection import RandomizedSearchCV
from itertools import product
import seaborn as sns
import matplotlib.pyplot as plt
from collections.abc import Iterable
import math
import random
from scipy.stats import spearmanr, pearsonr
import math
import copy

!pip install xlsxwriter
import xlsxwriter

pd.options.mode.chained_assignment = None  # default='warn'
!wget https://raw.githubusercontent.com/amirpandi/METIS/main/utils.py

from utils import *

#[20240229] section added to Echo section to assign the source well to its respective target well [by Bob van Sluijs]
#[20240306] modifications for rounding the volumes to multiples of 25 nL [by Leonardo Morini]
#[20240306] modifications to the Echo part developed by Bob: (i) intead of filling a single source well to the upper limit of 65 nL, the code now splits the voume equally among the wells containing the same reagent; (ii) source wells are assigned vertically intead of horizontally and it follow the sequence of reagents specified in the beginning of the book [by Bob van Sluijs]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Help for each imported function is available:

<h1>METIS</h1>
<h3>TBI Notebook</h3>
Before use, read figure 2, Supplementary Note 2 and Supplementary Figures 4-6 of the METIS publication at: <a>biorxiv.org/content/10.1101/2021.12.28.474323v2</a>

In [ ]:
help(random_combination_generator)

This notebook includes all data processing, active learning, results analysis and visualization.

# User Inputs

<p1><h2> When to use this part:</h2>
* every time before using MLOP you should fill this part based on your project</p1>
<p1><h2> How to use this part:</h2>
* User should upload all available Results file (i.e. Results_1.csv to Results_n.csv) to runtime from Files/UploadFile
* now there is an example, change it to your project</p1>

In [ ]:
# General Parameters:
m = 30  # number_of_combination_each_round
minimum_drop_size_nanoliter = 25
final_reaction_volume_nanoliter = 11000
maximum_volume_of_model_output = 10900 # (i. e., total volume minus fixed parts/components. Avoid 0 for water.)
fixed_parts = {} # 33% of total volume is Lysate, 15% energy mix (except variable one), 2.5% IPTG
days_total = 5 # how many days you want to continue # set it at your max prediction

# Model Parameters:
RandomCV = True
n_iter = 200
ensemble_len = 20
exploration = {1: 1.41, 2: 1.41, 3: 0.5,
               4: 0.5, 5: 1.0, 6: 1.41, 7: 1.0,
               8: 1.0, 9: 0.5, 10: 0.5}
days_range = [m for i in range(days_total)]

**metabolite name must not include "_"**


In [ ]:
# Part 1: choose grid for our metabolite conc and define stock concentration

# It is recommended to first define the min and max conc, run this cell,
# see the concentrations generated by the model that are proportional to the stock concentration and the minimum pipetting volume.
# then from the suggested range choose the desired ones in conc_value and finally set min and max to None.

# each metabolite min, max and stock must be in same units

# concentrations_limits :

concentrations_limits = {
 'hepes':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.5,1,2], 'Conc_Stock':33.3, 'Alternatives':None},
 'k-glut':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[3,6,12], 'Conc_Stock':100, 'Alternatives':None},
 'mg-acet':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[1,2,4], 'Conc_Stock':66.7, 'Alternatives':None},
 'gsh':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[1.25,2.5,5], 'Conc_Stock':80.0, 'Alternatives':None},
 'spermidine':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.1,0.2,0.4], 'Conc_Stock':6.7, 'Alternatives':None},
 'fthf':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.5,1,2], 'Conc_Stock':33.3, 'Alternatives':None},
 'cp':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[2,4,6], 'Conc_Stock':66.7, 'Alternatives':None},
 'aa1':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[5,10,20], 'Conc_Stock':100, 'Alternatives':None},
 'aa2':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[2,4,8], 'Conc_Stock':100, 'Alternatives':None},
 'aa3':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[1,2,4], 'Conc_Stock':50, 'Alternatives':None},
 'atp-gtp':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[3,6,12], 'Conc_Stock':100.0, 'Alternatives':None},
 'ctp-utp':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[1,2,4], 'Conc_Stock':66.7, 'Alternatives':None},
 'tRNA':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[1.75,3.5,7], 'Conc_Stock':100, 'Alternatives':None},
 'aaRS-mtf':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.1,0.2,0.4], 'Conc_Stock':6.7, 'Alternatives':None},
 'IFs':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.1,0.2,0.4], 'Conc_Stock':6.7, 'Alternatives':None},
 'EFs':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[1.5,2,4], 'Conc_Stock':66.7, 'Alternatives':None},
 'RFs':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.05,0.1,0.2], 'Conc_Stock':3.3, 'Alternatives':None},
 't7pol':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.05,0.1,0.2], 'Conc_Stock':3.3, 'Alternatives':None},
 'ndk-mk-ck-ppiase':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.05,0.1,0.2], 'Conc_Stock':3.3, 'Alternatives':None},
 'factors':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.05,0.1,0.2], 'Conc_Stock':3.3, 'Alternatives':None},
 'ribosomes':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[3.33,6.7,13.32], 'Conc_Stock':100.0, 'Alternatives':None},
 'DNA':{'Conc_Min':None, 'Conc_Max':None, 'Conc_Values':[0.1], 'Conc_Stock':2.80, 'Alternatives':None}} #Concentration in nM

In [ ]:
# Check Possible Concentrations
data_lists = {}
num = 0

pool_size = 1

for key, value in concentrations_limits.items():
    print('Possible Conc For :',key)
    if not value['Conc_Values']:
        print('Your Min, Max :', "({}, {})".format(value['Conc_Min'], value['Conc_Max']))
    concs, vols = allowed_output(value, reaction_vol_nl = final_reaction_volume_nanoliter, drop_size_nl = minimum_drop_size_nanoliter, verbose=0)
    print(concs)
    print(vols)

    pool_size *= len(concs)

    data_lists[num] = vols
    num += 1
    print()

print('All Possible Combination Number = ', pool_size)
if pool_size > 10000000:
    pool_size = 10000000
    print('Percentage calculation is not availbe duo to large pool size!')
else:
    percent , pool_size = percentage_possible(data_lists, threshold = maximum_volume_of_model_output)
    print(percent, f'% of {pool_size} possible combination are executable!')

In [ ]:
# add reference and negative control, these combinations will be implemented in concentrations and volume files
# leave it empty if you dont need, but still you need to run this cell
# you can add more desired combination to this dictionary
# *** conc in these dic must be compatible with minimum drop size ***, for help you can see our other examples on METIS Github repo

specials = {"REF": {'atp-gtp':[6],'ctp-utp':[2],'gsh':[2.5], 'EFs':[2], 'ribosomes':[6.66], 'hepes':[1], 'tRNA':[3.5],
                    't7pol':[0.1], 'mg-acet':[2], 'k-glut':[6], 'spermidine':[0.2],'fthf':[1],'aa1':[10], 'aa2':[4], 'aa3':[2],
                    'cp':[4],'aaRS-mtf':[0.2],'IFs':[0.2], 'RFs':[0.1],'ndk-mk-ck-ppiase':[0.1],'factors':[0.1], 'DNA':[0.1]},
            "Top_Day1": {'atp-gtp':[6],'ctp-utp':[4],'gsh':[2.5], 'EFs':[2], 'ribosomes':[3.33], 'hepes':[1], 'tRNA':[1.75],
                    't7pol':[0.2], 'mg-acet':[2], 'k-glut':[6], 'spermidine':[0.1],'fthf':[2],'aa1':[10], 'aa2':[8], 'aa3':[1],
                    'cp':[2],'aaRS-mtf':[0.4],'IFs':[0.1], 'RFs':[0.05],'ndk-mk-ck-ppiase':[0.1],'factors':[0.05], 'DNA':[0.1]},
            "Top_Day2": {'atp-gtp':[3],'ctp-utp':[4],'gsh':[2.5], 'EFs':[2], 'ribosomes':[3.33], 'hepes':[2], 'tRNA':[1.75],
                    't7pol':[0.2], 'mg-acet':[2], 'k-glut':[6], 'spermidine':[0.2],'fthf':[0.5],'aa1':[20], 'aa2':[8], 'aa3':[1],
                    'cp':[2],'aaRS-mtf':[0.1],'IFs':[0.2], 'RFs':[0.1],'ndk-mk-ck-ppiase':[0.05],'factors':[0.05], 'DNA':[0.1]},
            "Top_Day3": {'atp-gtp':[3],'ctp-utp':[4],'gsh':[5], 'EFs':[1.5], 'ribosomes':[6.7], 'hepes':[2], 'tRNA':[1.75],
                    't7pol':[0.2], 'mg-acet':[2], 'k-glut':[6], 'spermidine':[0.2],'fthf':[1],'aa1':[5], 'aa2':[8], 'aa3':[1],
                    'cp':[2],'aaRS-mtf':[0.2],'IFs':[0.4], 'RFs':[0.05],'ndk-mk-ck-ppiase':[0.1],'factors':[0.05], 'DNA':[0.1]},
            "Top_Day4": {'hepes':[0.5],'k-glut':[3.0],'mg-acet':[2.0],'gsh':[5.0],'spermidine':[0.1],'fthf':[2.0],'cp':[2.0],
                      'aa1':[20.0],'aa2':[8.0],'aa3':[2.0],'atp-gtp':[3.0],'ctp-utp':[1.0],'tRNA':[1.75],'aaRS-mtf':[0.4],'IFs':[0.4],
                      'EFs':[1.5],'RFs':[0.05],'t7pol':[0.1],'ndk-mk-ck-ppiase':[0.05],'factors':[0.1],'ribosomes':[6.7],'DNA':[0.1]},
            }

# Day 1 (in absence of Day_0)

<p1><h2> When to use this part:</h2>
* after filling "User Input" if you have no pre existing data (Day_0.csv)
<p1><h2> How to use this part:</h2>
* just run all cells! and download the output concentration and volume files from the folder at left.


In [ ]:
# make random combinations
Concentrations_1 = random_combination_generator(concentrations_limits, number_of_combination = m,
                                                reaction_vol_nl=final_reaction_volume_nanoliter,
                                                max_nl=maximum_volume_of_model_output, drop_size_nl=minimum_drop_size_nanoliter, return_df=True)

# add control, reference and other desired combinations
df_specials = [pd.DataFrame(i) for i in specials.values()]
Concentrations_1 = pd.concat([Concentrations_1, *df_specials]).reset_index(drop=True)

Concentrations_1

In [ ]:
!mkdir -p Day_1
Concentrations_1.to_csv('Day_1/Concentrations_1.csv', index=False)

In [ ]:
# concentration_to_volume
Volumes_1 = round(concentration_to_volume(Concentrations_1, concentrations_limits, reaction_mixture_vol_nl=final_reaction_volume_nanoliter,
                                    fixed_parts=fixed_parts)/25)*25
Volumes_1.to_csv('Day_1/Volumes_1.csv', index=False)
Volumes_1

# Other Days

<p1><h2> When to use this part:</h2>
* when you have either pre existing data (Results_0.csv) or Results_1 to Results_n uploaded into the main directory at left.
<p1><h2> How to use this part:</h2>
* just run all cells!

In [ ]:
# day = day_finder('Results') - 1
# start_day = (1, 0)[os.path.isfile('Results_0')]
# print(start_day, day)

In [ ]:
start_day = 0
day = 4

In [ ]:
desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day+1):
    if start_day or num>0:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

if 'reference' in specials.keys():
    ref_data = pd.DataFrame(specials['reference'])
    ref_label = pd.DataFrame({'yield':[1.0]})
    aggregated_data_m = pd.concat([aggregated_data_m, ref_data[desired_cols]]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, ref_label]).reset_index(drop=True)

aggregated_data_m

In [ ]:
# ensemble of regressors
# XGBRegressor is an enhanced random forest boosted algorithm
RandomCV = True
if RandomCV:
    # create a default XGBoost classifier
    model = XGBRegressor(objective = 'reg:squarederror')

    # Create the grid search parameter and scoring functions
    param_grid = {
        "learning_rate": [0.01, 0.03, 0.1, 0.3],
        "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
        "subsample": [0.6, 0.8, 0.9, 1.0],
        "max_depth": [2, 3, 4, 6 , 8],
        "n_estimators": [10, 20,  40, 60, 80, 100, 300, 500],
        "reg_lambda": [1, 1.5, 2],
        "gamma": [0, 0.1, 0.4, 0.6],
        "min_child_weight": [1, 2, 4]}

    # create the grid search object
    grid = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=5,
        scoring= 'neg_mean_absolute_error',
        n_jobs=-1,
        n_iter=n_iter)

    print('RandomSearchCV ...')
    grid.fit(aggregated_data_m.values, aggregated_label_m.values)
    results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
    regressors_list = [XGBRegressor(
                       objective = 'reg:squarederror',**param) for param in results.params.iloc[0:ensemble_len,] ]
    print('RandomSearchCV Done!')

else:
    # Default
    regressors_list = [XGBRegressor(
        objective = 'reg:squarederror',
        n_estimators = n,
        learning_rate = l ,
        max_depth = 6,
        min_child_weight = 4,
        subsample = 0.9,
        gamma = 0.4,
        colsample_bytree = 0.9) for n in (10, 20, 30, 40, 50) for l in (0.01, 0.03, 0.1, 0.2)]

In [ ]:
pool_size

In [ ]:
t0 = time.time()

pool_size = 10000 # for when you want to change pool size, default = all possible combinations

Concentrations_n_m = bayesian_optimization(regressors_list, aggregated_data_m, aggregated_label_m, concentrations_limits,
                                           final_order=final_order,
                                           df_main = aggregated_data_m,
                                           reaction_vol_nl=final_reaction_volume_nanoliter, max_nl=maximum_volume_of_model_output,
                                           drop_size_nl=minimum_drop_size_nanoliter,
                                           exploitation=1, exploration=exploration[day+1], test_size=m, pool_size=pool_size, verbose=0,
                                           day=day, days_range = days_range)

print("Passed Time(s): ",time.time()-t0)

Concentrations_n_m

In [ ]:
# add control, reference and other desired combinations
df_specials = [pd.DataFrame(i) for i in specials.values()]
Concentrations_n = pd.concat([Concentrations_n_m, *df_specials]).reset_index(drop=True)

name_folder = 'Day_{}'.format(day+1)
! mkdir -p {name_folder}

Concentrations_n.to_csv('Day_{}/Concentrations_{}.csv'.format(day+1, day+1), index=False)

In [ ]:
# check not to make repeated combination
df_main = aggregated_data_m

comparison_df = df_main.merge(pd.read_csv('Day_{}/Concentrations_{}.csv'.format(day+1, day+1)).iloc[:m,:],
                              indicator=True,
                              how='outer')

comparison_df._merge.unique()

In [ ]:
# concentration_to_volume
Volumes_n = round(concentration_to_volume(Concentrations_n, concentrations_limits, reaction_mixture_vol_nl=final_reaction_volume_nanoliter, fixed_parts=fixed_parts)/25)*25

Volumes_n.to_csv('Day_{}/Volumes_{}.csv'.format(day+1, day+1), index=False)

Modification of Concentrations_n.csv possible and (= Custom file)
Can then be converted to Volumes_n.csv using the following cell

In [ ]:
###
## Custum Round : Cell used only if you have a custom Concentrations_n.csv file
###

name_folder = 'Day_{}'.format(day + 1)

# Load your Concentrations_n.csv file based on the current day+1
concentrations_file = '{}/Concentrations_{}.csv'.format(name_folder, day + 1)
Concentrations_n = pd.read_csv(concentrations_file)

# Perform the conversion from concentrations to volumes
Volumes_n = round(concentration_to_volume(Concentrations_n, concentrations_limits, reaction_mixture_vol_nl=final_reaction_volume_nanoliter, fixed_parts=fixed_parts) / 25) * 25

# Create the output folder if it doesn't exist
! mkdir -p {name_folder}

# Save the resulting Volumes_n to a new CSV file
Volumes_n.to_csv('{}/Volumes_{}.csv'.format(name_folder, day + 1), index=False)

print("Volumes_{}.csv generated successfully!".format(day + 1))



# Visualising Results as BoxPlot


<p1><h2> When to use this part:</h2>
* you have at least one Results.csv file
<p1><h2> How to use this part:</h2>
* just run all cells!
* you can change plotting parameter

In [ ]:
display_std = False # if you have included "yield_std" column in Results.csv will show it else disply 0 for std

day = day_finder('Results')
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

Results_m = pd.DataFrame(columns=['yield', 'std', 'Day'])
for i in range(start_day, day):
    Results_i = pd.DataFrame(columns=['yield', 'std', 'Day'])
    if start_day:
        Results_i['yield'] = pd.read_csv(f'Results_{i}.csv')['yield'].iloc[0:days_range[i-1]]
    else:
        Results_i['yield'] = pd.read_csv(f'Results_{i}.csv')['yield'].iloc[0:len(pd.read_csv('Results_0.csv'))]

    try:
        if start_day:
            Results_i['std'] = pd.read_csv(f'Results_{i}.csv')['yield_std'].iloc[0:days_range[i-1]]
        else:
            Results_i['std'] = pd.read_csv(f'Results_{i}.csv')['yield_std'].iloc[0:len(pd.read_csv('Results_0.csv'))]

    except:
        Results_i['std'] = 0
    Results_i['Day'] = i
    Results_m = pd.concat([Results_m, Results_i])

plt.style.use('seaborn-whitegrid')
plt.style.use('seaborn-poster')
fig, ax = plt.subplots(1, 1)

#ax = sns.boxplot(x='Day', y='yield', data=Results_m, color='#4BC15F', fliersize=0)
ax = sns.boxplot(x='Day', y='yield', data=Results_m, color='#00A368', fliersize=0)

ax = sns.swarmplot(x='Day', y='yield', data=Results_m, color='k', size=10)

if display_std:
    # Find the x,y coordinates for each point
    order = 0
    for point_pair in ax.collections:
        for x, y in point_pair.get_offsets():
            std = Results_m.sort_values(['Day', 'yield'], ignore_index=True)['std'][order]
            y_Result = Results_m.sort_values(['Day', 'yield'], ignore_index=True)['yield'][order]
            ax.arrow(x, y-(std/2), dx=0, dy=std, linewidth=1, zorder=4, width=0, color='grey')

            order += 1

    errors = Results_m['std']

ax.set_ylim(top=10)
ax

SAVE plots and download from the directory at left

In [ ]:
fig.savefig(f'#00A368_Day_{day-1}_Boxplot.png', format='png', dpi=1200)
fig.savefig(f'#00A368_Day_{day-1}_Boxplot.svg', format='svg', dpi=1200)

# Visualising Results For Each Metabolite

<p1><h2> When to use this part:</h2>
* you have at least one Results.csv file
<p1><h2> How to use this part:</h2>
* just run all cells!
* you can change plotting parameter

In [ ]:
desired_cols = []
day = day_finder('Results')
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day):
    if start_day:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

plt.style.use('seaborn-whitegrid')

num = len(desired_cols)
dim = math.ceil(math.sqrt(num))
data = pd.concat([aggregated_data_m, aggregated_label_m], axis=1)

import matplotlib as mpl


mpl.rcParams['axes.linewidth'] = 8.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally
mpl.rcParams['xtick.major.pad'] = 10.0 #set the value globally
mpl.rcParams['ytick.major.pad'] = 10.0 #set the value globally

fig = plt.figure(figsize=(25,20))

for i in range(1, num+1):
    ax = plt.subplot(dim, dim, i)
    ax.xaxis.grid(False)
    ax = sns.scatterplot(x = desired_cols[i-1], y='yield', color='darkblue', data=data, s=120, linewidth=0)
    y_axis = ax.axes.get_yaxis()
    y_axis.set_label_text('foo')
    y_label = y_axis.get_label()
    ##print isinstance(x_label, matplotlib.artist.Artist)
    y_label.set_visible(False)
    #ax.xaxis.set_tick_params()
    plt.xticks(weight = 'bold', fontsize=18)
    plt.yticks(weight = 'bold', fontsize=18)

    plt.xlabel(desired_cols[i-1], labelpad=30 ,fontsize=25, weight='bold')


fig.tight_layout(h_pad=3, w_pad=3)

SAVE plots and download from the directory at left

In [ ]:
fig.savefig(f'Day_{day-1}_Metabolite_Yield.png', format='png', dpi=1200)
fig.savefig(f'Day_{day-1}_Metabolite_Yield.svg', format='svg', dpi=1200)

# Visualising Results For Each Metabolite (Beta)

In [ ]:
desired_cols_alternatives = []
desired_cols = []
alternative_dict = {}

day = day_finder('Results')
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols_alternatives += alternative_name
        alternative_dict[key] = alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols + desired_cols_alternatives)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day):
    if start_day:
        data_m, label_m, _, _= result_preprocess(num, desired_cols + desired_cols_alternatives, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols + desired_cols_alternatives, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

In [ ]:
plt.style.use('seaborn-whitegrid')

num = len(desired_cols)
dim = math.ceil(math.sqrt(num))
data = pd.concat([aggregated_data_m, aggregated_label_m], axis=1)

import matplotlib as mpl


mpl.rcParams['axes.linewidth'] = 8.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally
mpl.rcParams['xtick.major.pad'] = 10.0 #set the value globally
mpl.rcParams['ytick.major.pad'] = 10.0 #set the value globally

fig = plt.figure(figsize=(25,20))

for i in range(1, num+1):
    ax = plt.subplot(dim, dim, i)
    ax.xaxis.grid(False)
    ax = sns.scatterplot(x = desired_cols[i-1], y='yield', color='darkblue', data=data, s=120, linewidth=0)
    y_axis = ax.axes.get_yaxis()
    y_axis.set_label_text('foo')
    y_label = y_axis.get_label()
    ##print isinstance(x_label, matplotlib.artist.Artist)
    y_label.set_visible(False)
    #ax.xaxis.set_tick_params()
    plt.xticks(weight = 'bold', fontsize=18)
    plt.yticks(weight = 'bold', fontsize=18)

    plt.xlabel(desired_cols[i-1], labelpad=30 ,fontsize=25, weight='bold')


fig.tight_layout(h_pad=3, w_pad=3)

SAVE plots and download from the directory at left

In [ ]:
fig.savefig(f'Day_{day-1}_Metabolite_Yield_NoAlternative.png', format='png', dpi=1200)
fig.savefig(f'Day_{day-1}_Metabolite_Yield_NoAlternative.svg', format='svg', dpi=1200)

In [ ]:
temp_dict = {}
for key, value in alternative_dict.items():
    temp_dict[key] = {}
    for i in value:
        temp_dict[key][i] = []

data = pd.concat([aggregated_data_m, aggregated_label_m], axis=1)
for i in range(len(data)):
    for j in temp_dict.keys():
        for k in temp_dict[j].keys():
            if  data[desired_cols_alternatives].loc[i][k]:
                temp_dict[j][k].append(aggregated_label_m.loc[i]['yield'])

plt.style.use('seaborn-whitegrid')

num = len(temp_dict.keys())
dim = math.ceil(math.sqrt(num))

import matplotlib as mpl

mpl.rcParams['axes.linewidth'] = 8.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally
mpl.rcParams['xtick.major.pad'] = 10.0 #set the value globally
mpl.rcParams['ytick.major.pad'] = 10.0 #set the value globally

fig = plt.figure(figsize=(15,10))

for ind, main in enumerate(temp_dict.keys()):
    ax = plt.subplot(dim, dim, ind+1)
    temp_df = pd.DataFrame(columns=temp_dict[main].keys(), index=range(max([len(i) for i in temp_dict[main].values()])))
    for i in temp_dict[main].keys():
        temp_df[i].loc[0:len(temp_dict[main][i])-1] = temp_dict[main][i]

    sns.swarmplot(data=temp_df, ax=ax, size=10, palette=sns.color_palette(['goldenrod', 'royalblue' , 'firebrick', 'black']))


fig.tight_layout(h_pad=3, w_pad=3)

SAVE plots and download from the directory at left

In [ ]:
fig.savefig(f'Day_{day-1}_Metabolite_Yield_Alternative.png', format='png', dpi=1200)
fig.savefig(f'Day_{day-1}_Metabolite_Yield_Alternative.svg', format='svg', dpi=1200)

# Viualising Concentrations From Day_1 to Now

<p1><h2> When to use this part:</h2>
* you have at least one Results.csv file
<p1><h2> How to use this part:</h2>
* just run all cells!
* you can change plotting parameter

In [ ]:
data_all = pd.DataFrame(columns = list(concentrations_limits.keys())+['day'])
day = day_finder('Results')
desired_cols = []

# make column name:
desired_cols = []
for key, value in concentrations_limits.items():
    if not value['Alternatives']:
        desired_cols.append(key)
    else:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name

final_order = desired_cols

d = (1, 0)[os.path.isfile('Results_0.csv')]

while os.path.exists(f'Results_{d}.csv'):
    data = pd.read_csv(f'Results_{d}.csv')
    data = data[desired_cols]
    data = data.iloc[:days_range[d-1],:]
    data['day'] = f'Day_{d}'
    data_all = pd.concat([data_all, data]).reset_index(drop=True)
    d += 1

import matplotlib as mpl


mpl.rcParams['axes.linewidth'] = 8.0 #set the value globally
mpl.rcParams['axes.edgecolor'] = 'black' #set the value globally
mpl.rcParams['xtick.major.pad'] = 10.0 #set the value globally


num = len(desired_cols)
dim = math.ceil(math.sqrt(num))

fig = plt.figure(figsize=(25,20))
for i in range(1, num+1):
    ax = plt.subplot(dim, dim, i)
    ax = sns.stripplot(x=desired_cols[i-1], y="day", color='darkblue', data=data_all, s=10)
    y_axis = ax.axes.get_yaxis()
    y_axis.set_label_text('foo')
    y_label = y_axis.get_label()
    y_label.set_visible(False)

    ax.yaxis.set_ticklabels([])
    plt.xticks(weight = 'bold', fontsize=18)

    plt.xlabel(desired_cols[i-1], labelpad=30 ,fontsize=25, weight='bold')


fig.tight_layout(h_pad=3, w_pad=3)

In [ ]:
data_all

In [ ]:
fig.savefig(f'Day_{day-1}_Metabolite_Days.png', format='png', dpi=1200)
fig.savefig(f'Day_{day-1}_Metabolite_Days.svg', format='svg', dpi=1200)

# Transform Volumes.csv to Table2Speech Compatible Input

<p1><h2> When to use this part:</h2>
* you have at least one Volumes_n.csv file located in Day_n Folder
<p1><h2> How to use this part:</h2>
* specify in "which_day" your Volumes file is located
* run all other cells!
* you can download the Table2Speech_Volumes_n.csv file from Files

In [ ]:
which_day = 5 # i.e. Volumes_1.csv is in Day_1 Folder

In [ ]:
data = pd.read_csv(f'Day_{which_day}/Volumes_{which_day}.csv')

final_columns = []
columns_name = data.columns
for column in columns_name:
    if "_" in column:
        column_value = column.split('_')[0]
        column_to_add = data[column].unique()
        df_to_add = pd.DataFrame(columns = column_to_add,  data=np.zeros((len(data),len(column_to_add))))
        for i in range(len(data)):
            df_to_add[data[column].iloc[i,]][i] = data[column_value].iloc[i,]
        data = pd.concat([data, df_to_add], axis=1)
        final_columns.remove(column_value)
        final_columns += list(column_to_add)
    else:
        final_columns.append(column)

data = data[final_columns]
data

SAVE and download from the directory at left

In [ ]:
data.to_csv(f'Table2Speech_Volumes_{which_day}.csv', index=False)

# Transform Volumes.csv to ECHO liquid Handler Compatible Input

<p1><h2> When to use this part:</h2>
* you have at least one Volumes_n.csv file located in Day_n Folder
<p1><h2> How to use this part:</h2>
* specify in "which_day" your Volumes file is located
* run all other cells!
* you can download the Echo_Input_n.csv file from Files
* After generating and downloading the file you have to fill your source wells manually, depending on the total volume needed you probably need a number of different source wells

In [ ]:
which_day = 5 # i.e. Volumes_1.csv is in Day_1 Folder
plate_384_well = True  # your destination plate. If False the destination template will be 96-well plate.
triplicate = True  # Set to False to prevent triplicate handling
duplicate = False  # Add this line to handle duplicates
vertical = True  # vertical = True means it will fill the plate by column by column and vertical = False mean filling the plate row by row

In [ ]:
data = pd.read_csv(f'Day_{which_day}/Volumes_{which_day}.csv')

final_columns = []
columns_name = data.columns
for column in columns_name:
    if "_" in column:
        column_value = column.split('_')[0]
        column_to_add = data[column].unique()
        df_to_add = pd.DataFrame(columns = column_to_add,  data=np.zeros((len(data),len(column_to_add))))
        for i in range(len(data)):
            df_to_add[data[column].iloc[i,]][i] = data[column_value].iloc[i,]
        data = pd.concat([data, df_to_add], axis=1)
        final_columns.remove(column_value)
        final_columns += list(column_to_add)
    else:
        final_columns.append(column)

data = data[final_columns]
data

In [ ]:
intermediate = put_volumes_to_wells(data, plate_384_well=plate_384_well, vertical=vertical, triplicate=triplicate, starting_well='A1', make_csv=False)
intermediate

In [ ]:
echo_input = source_to_destination(intermediate, desired_order=None, reset_index=True, check_zero=False)
echo_input[1]

In [ ]:
echo_input[1].to_csv(f'Echo_Input_{which_day}.csv', index=False)

In [ ]:
echo_input[1].sort_values(by='Transfer_Volume')

In [ ]:
##
## Converts incorrect ouptut file into Echo Cherry picking csv file
## 05/12/2024 : Corrected "Destination Plate Barcode" to " Destination Plate Name"
########

csv_file = f'Echo_Input_{day+1}.csv'

csv_out_name = f'Echo_Cherry_Picking_{day+1}.csv'
csv_echo = pandas.read_csv(csv_file)

#si en script python
# if len(sys.argv) != 2:
#     print('Usage: script_echo_to_cherry.py Echo_Input_X.csv')
#     sys.exit()
# csv_echo = pandas.read_csv(sys.argv[1])

csv_echo['Destination Plate Type'] = 'Greiner384_black_PS_Fbottom_781906'
csv_echo.rename(columns={"Destination_Plate_Barcode": "Destination Plate Name"}, inplace=True)
csv_echo.rename(columns={"Destination_Well": "Destination Well"}, inplace=True)
csv_echo.rename(columns={"Transfer_Volume": "Transfer Volume"}, inplace=True)
csv_echo["Sample name"] = csv_echo["Source_Well"]

csv_echo = csv_echo.reindex(['Destination Plate Type', 'Destination Plate Name', 'Sample name', 'Destination Well', 'Transfer Volume', 'Source Well', 'Source Plate Name', 'Source Plate Type'], axis=1)
csv_echo["Source Plate Name"] = "source1"
csv_echo["Destination Plate Name"] = "destination1"

csv_echo_water = csv_echo[csv_echo['Sample name'] == 'water well']
csv_echo_other = csv_echo[csv_echo['Sample name'] != 'water well']
csv_echo = pd.concat([csv_echo_water, csv_echo_other])
csv_echo.reset_index(drop=True, inplace=True)


def update_source_plate_type(row):
    if row['Sample name'] == 'water well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'hepes well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'k-glut well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 'mg-acet well':
        return '384PP_Plus_AQ_SP'
    elif row['Sample name'] == 'gsh well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'spermidine well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'fthf well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 'cp well':
        return '384PP_Plus_AQ_GPSB'
    elif row['Sample name'] == 'aa1 well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'aa2 well':
        return '384PP_Plus_AQ_SP'
    elif row['Sample name'] == 'aa3 well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'atp-gtp well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'ctp-utp well':
        return '384PP_Plus_AQ_SP'
    elif row['Sample name'] == 'tRNA well':
        return '384PP_Plus_AQ_BP'
    elif row['Sample name'] == 'aaRS-mtf well':
        return '384PP_Plus_AQ_GPSA'
    elif row['Sample name'] == 'IFs well':
        return '384PP_Plus_AQ_GPSA'
    elif row['Sample name'] == 'EFs well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 'RFs well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 't7pol well':
        return '384PP_Plus_AQ_GPSB'
    elif row['Sample name'] == 'ndk-mk-ck-ppiase well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 'factors well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 'ribosomes well':
        return '384PP_Plus_AQ_GP'
    elif row['Sample name'] == 'DNA well':
        return '384PP_Plus_AQ_BP'
    else:
        print(row['Source Plate Type'])
        return row['Source Plate Type']

csv_echo['Source Plate Type'] = csv_echo.apply(update_source_plate_type, axis=1)

csv_echo.to_csv(csv_out_name, index=False)

In [ ]:


## Volume Calculator Module (Experimental setup)


# (27/11/24) Added Determine replication factor based on duplicate and triplicate status
if duplicate and triplicate:
    raise ValueError("Both 'duplicate' and 'triplicate' are True. Please set only one to True.")
elif duplicate:
    replicate = 2
elif triplicate:
    replicate = 3
else:
    replicate = 1


# Read in the CSV file

volumes_file = f"Day_{day + 1}/Volumes_{day + 1}.csv"
df = pd.read_csv(volumes_file)

# Component-specific parameters
extra_volumes = {
    'hepes': 5, 'k-glut': 5, 'mg-acet': 4, 'gsh': 5, 'spermidine': 4, 'fthf': 4,
    'cp': 3, 'aa1': 8, 'aa2': 7, 'aa3': 5, 'atp-gtp': 6, 'ctp-utp': 5,
    'tRNA': 5, 'aaRS-mtf': 7, 'IFs': 5, 'EFs': 10, 'RFs': 6, 't7pol': 6,
    'ndk-mk-ck-ppiase': 7, 'factors': 6, 'ribosomes': 8, 'DNA': 3, 'water': 3
}

tube_limits = {
    'hepes': 120, 'k-glut': 240, 'mg-acet': 120, 'gsh': 125, 'spermidine': 120,
    'fthf': 120, 'cp': 240, 'aa1': 520, 'aa2': 160, 'aa3': 160, 'atp-gtp': 240,
    'ctp-utp': 120, 'tRNA': 140, 'aaRS-mtf': 120, 'IFs': 120, 'EFs': 120,
    'RFs': 120, 't7pol': 120, 'ndk-mk-ck-ppiase': 120, 'factors': 120,
    'ribosomes': 266.4, 'DNA': 1000, 'water': 1000
}

def calculate_volumes(df, replicate):
    """
    Calculate required volumes for each component.
    :param df: DataFrame with component volumes in nanoliters.
    :param replicate: The replication factor (1 for single, 2 for duplicate, 3 for triplicate)
    :return: DataFrame with calculated volumes and well requirements
    """
    df = df.sum(axis=0)  # Sum volumes for each component
    df /= 1000  # Convert from nanoliters to microliters
    df *= replicate  # Multiply for duplicates/triplicates

    # Calculate total volumes including extra and dead volumes
    total_volumes = {}
    wells_required = {}
    extra_allocated = {}  # Dictionary to hold extra volume allocations

    for component, volume in df.items():
        # Calculate number of source wells needed
        wells = np.ceil(volume / 45)
        wells_required[component] = wells

        # Add dead volume and extra volume
        extra_volume = extra_volumes[component]
        total_volume = volume + wells * 20 + wells * extra_volume
        total_volumes[component] = total_volume

        # Calculate volume per well
        volume_per_well = total_volume / wells

        # Check if the volume per well exceeds the dispenser limit (69 µl)
        if volume_per_well > 69:
            wells += 1  # Increase wells if it exceeds the limit
            total_volume = volume + wells * 20 + wells * extra_volume  # Recalculate total volume

        # Update the dictionaries with new values
        wells_required[component] = wells
        total_volumes[component] = total_volume
        extra_allocated[component] = extra_volume  # Store extra volume allocated

    # Calculate volume per well again after adjustments
    volume_per_well = {component: total_volumes[component] / wells_required[component] for component in total_volumes}

    return pd.DataFrame({
        'Volume (µl)': df,
        'Extra (µl)': extra_allocated,
        'Wells Required': wells_required,
        'Total Volume (with extras and dead volume)': total_volumes,
        'Volume per Well (µl)': volume_per_well
    })

def check_tube_limits(df):
    """
    Check if the volumes are within the limits that can be prepared per tube.
    :param df: DataFrame with total volumes.
    :return: DataFrame indicating if the volume exceeds tube limits.
    """
    limits_exceeded = {}
    for component, volume in df['Total Volume (with extras and dead volume)'].items():
        if volume > tube_limits[component]:
            limits_exceeded[component] = "Exceeds tube limit"
        else:
            limits_exceeded[component] = "Within limit"

    df['Tube Limit Check'] = limits_exceeded
    return df

volumes_df = calculate_volumes(df, replicate=2)
result_df = check_tube_limits(volumes_df)

# Save the result as an Excel file
output_file = f"Day_{day + 1}/Volume_Calculator_{day + 1}.xlsx"
result_df.to_excel(output_file, index=True)

print(result_df)
print(f"Results saved to {output_file}")


Create Source Plate Layout from Volume Calculator

In [ ]:
import pandas as pd
import numpy as np
import openpyxl
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment

def create_well_plate_layout(df, output_file):
    # Create a 384-well plate layout (16 rows x 24 columns)
    plate_rows = 16
    plate_cols = 24

    # Prepare an empty DataFrame to represent the 384-well plate
    plate = pd.DataFrame('', index=[chr(i) for i in range(65, 65 + plate_rows)], columns=range(1, plate_cols + 1))

    # Fill the plate with component names and volumes, starting from top-left
    row_idx, col_idx = 0, 0
    components_in_row = 0  # Count unique components in the current row

    for component, data in df.iterrows():
        wells = int(data['Wells Required'])
        volume = data['Volume per Well']

        # Write component and volume into wells
        for i in range(wells):
            row = chr(65 + row_idx)
            col = col_idx + 1
            plate.at[row, col] = f"{component} well {volume:.1f}"

            components_in_row += 1

            col_idx += 1
            if col_idx == plate_cols:  # Move to next row when current row is filled
                col_idx = 0
                row_idx += 1
                components_in_row = 0  # Reset component count for new row

            # Move to next row if we have 5 unique components
            if components_in_row >= 5 and (i == 0):  # Only check at the start of the component
                col_idx = 0
                row_idx += 1
                components_in_row = 0  # Reset for the next row

    # Write the well plate layout to an Excel file with formatting
    with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
        plate.to_excel(writer, index=True, header=True, sheet_name="384-Well Plate")
        workbook = writer.book
        worksheet = writer.sheets["384-Well Plate"]

        # Set column widths and row heights
        for col in range(1, plate_cols + 2):  # +2 to account for Excel column indexing (starts from 1)
            column_letter = get_column_letter(col)
            worksheet.column_dimensions[column_letter].width = 10  # Set small width for empty cells

        for row in range(1, plate_rows + 2):  # +2 to account for row indexing
            worksheet.row_dimensions[row].height = 15  # Set small height for empty cells

        # Adjust size only for cells containing components
        for row in worksheet.iter_rows(min_row=2, max_row=plate_rows+1, min_col=2, max_col=plate_cols+1):
            for cell in row:
                if cell.value:  # Check if the cell is not empty
                    cell.alignment = Alignment(horizontal='center', vertical='center')
                    cell.font = openpyxl.styles.Font(size=10)  # Set font size
                    # Set larger dimensions for cells with component values
                    worksheet.row_dimensions[cell.row].height = 40  # Increase height for component cells
                    worksheet.column_dimensions[get_column_letter(cell.column)].width = 20  # Increase width

result_df['Volume per Well'] = result_df['Total Volume (with extras and dead volume)'] / result_df['Wells Required']



# File path to save the formatted Excel file
output_file = f"Day_{day + 1}/Source_Plate_{day + 1}.xlsx"

# Create the well plate layout with formatted cells
create_well_plate_layout(result_df, output_file)

print(f"Formatted 384-well plate layout saved to: {output_file}")


Fill the "Source Well3 column of the cherry picking file based on the source plate layout

In [ ]:
import pandas as pd

# Load your Echo Cherry Picking CSV file
csv_file = f'Echo_Cherry_Picking_{day + 1}.csv'
csv_echo = pd.read_csv(csv_file)

# Load the source plate layout from the Excel file
layout_file = f"Day_{day + 1}/Source_Plate_{day + 1}.xlsx"
source_plate_layout = pd.read_excel(layout_file, sheet_name='384-Well Plate', index_col=0)

# Flatten the source plate layout for easier matching
source_well_map = {}
for row in source_plate_layout.index:
    for col in source_plate_layout.columns:
        component = source_plate_layout.at[row, col]
        if pd.notna(component):  # Only map non-empty cells
            component_name = component.split(' well')[0]  # Extract component name
            if component_name not in source_well_map:
                source_well_map[component_name] = []
            source_well_map[component_name].append(f"{row}{col}")

# Debugging: Print the source well map
print("Source Well Map:")
for component, wells in source_well_map.items():
    print(f"{component}: {wells}")

# Function to assign source wells in blocks based on component distribution
def assign_source_well(row):
    component = row['Sample name'].split(' well')[0]  # Extract component name
    if component in source_well_map:
        well_list = source_well_map[component]
        # Count the number of occurrences of the component in the CSV
        component_count = csv_echo['Sample name'].value_counts().get(row['Sample name'], 0)
        num_samples = component_count // len(well_list)  # Calculate samples per well
        if num_samples == 0:
            return well_list[0]  # If there are no samples, assign the first well
        well_index = (row.name // num_samples) % len(well_list)  # Fill wells in blocks
        return well_list[well_index]
    else:
        print(f"Warning: '{component}' not found in source well map.")  # Debugging statement
        return 'Unknown'

# Apply the source well assignment
csv_echo['Source Well'] = csv_echo.apply(assign_source_well, axis=1)

# Debugging: Check the first few rows of the updated DataFrame
print("Updated CSV Echo with Source Wells:")
print(csv_echo.head())

# Save the updated file
output_file = f'Echo_Cherry_Picking_Filled_{day + 1}.csv'
csv_echo.to_csv(output_file, index=False)

print(f"Updated Echo Cherry Picking file saved to: {output_file}")


# Find K Most Informative Combinations

<p1><h2> When to use this part:</h2>
* you have at least one Results.csv file </p1>
<p1><h2> How to use this part:</h2>
* specify in "k" i.e. how many combinations you want to get
* run all other cells!
* you can download K_Most_Informative_Combinations.csv file from Files that contain index of most informative combinations for 5 iterations</p1>

In [ ]:
k = 20
number_try = 2000
number_to_select = 5
n_iter = 200

In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day + 1):
    if start_day or num>0:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)


aggregated_data_m

In [ ]:
# Create the grid search parameter and scoring functions
param_grid = {
        "learning_rate": [0.01, 0.03, 0.1, 0.3],
        "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
        "subsample": [0.6, 0.8, 0.9, 1.0],
        "max_depth": [2, 3, 4, 6 , 8],
        "n_estimators": [10, 20,  40, 60, 80, 100, 300, 500],
        "reg_lambda": [1, 1.5, 2],
        "gamma": [0, 0.1, 0.4, 0.6],
        "min_child_weight": [1, 2, 4]}

model = XGBRegressor(objective = 'reg:squarederror')
# create the grid search object
grid = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=5,
        scoring= 'neg_mean_absolute_error',
        n_jobs=-1,
        n_iter=n_iter)

print('RandomSearchCV ...')
grid.fit(aggregated_data_m.values, aggregated_label_m.values)
results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
print('RandomSearchCV Done!')

In [ ]:
indexs = []
num = 0
while num < number_try:
    numbers = set()
    while len(numbers) < k:
        numbers.add(random.randint(0, len(aggregated_data_m)-1))
    indexs.append(numbers)
    num += 1

In [ ]:
performance = []
best_param = results.params.iloc[0,]
all_set = set(range(len(aggregated_data_m)))
counter = 0
for index in indexs:
    counter += 1
    if counter%100 == 0:print(counter)
    index_test = all_set - index
    index_test = list(index_test)
    index = list(index)
    model = XGBRegressor(objective = 'reg:squarederror', **best_param)
    model.fit(aggregated_data_m.iloc[index].values, aggregated_label_m.iloc[index].values)
    performance.append(spearmanr(model.predict(aggregated_data_m.iloc[index_test].values), aggregated_label_m.iloc[index_test].values).correlation**2)

performance = np.array(performance)

In [ ]:
ind = np.argpartition(performance, -number_to_select)[-number_to_select:]
performance[ind]

In [ ]:
plt.scatter(range(len(performance)), performance, s=3)

In [ ]:
df_perfomance = pd.DataFrame({'performance':performance, 'index_values':indexs}).sort_values('performance', ascending=False)
df_perfomance.iloc[0:5, :]

In [ ]:
df_perfomance.iloc[0:5, :].index_values

In [ ]:
for i, index in enumerate(df_perfomance.iloc[0:5, :].index_values):
    aggregated_data_m.iloc[list(index),:].to_csv(f'{k}_Most_Informative_Combinations_Rank{i+1}.csv', index=False)

# Find Feature Importances


<p1><h2> When to use this part:</h2>
* you have at least one Results.csv file </p1>
<p1><h2> How to use this part:</h2>
* run all cells!
* you can download importance_df.csv  from Files</p1>

In [ ]:
n_iter = 20

In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

In [ ]:
importance_list = []
print('This may take minutes!')

for day_i in range(1, day+1):
    aggregated_data_m = pd.DataFrame(columns=desired_cols)
    aggregated_label_m = pd.DataFrame(columns=['yield'])

    for num in range(start_day, day_i + 1):
        if start_day or num>0:
            data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
        else:
            data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

        aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
        aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)

    # Create the grid search parameter and scoring functions
    param_grid = {
        "learning_rate": [0.01, 0.03, 0.1, 0.3],
        "colsample_bytree": [0.6, 0.8, 0.9, 1.0],
        "subsample": [0.6, 0.8, 0.9, 1.0],
        "max_depth": [2, 3, 4, 6 , 8],
        "n_estimators": [10, 20,  40, 60, 80, 100, 300, 500],
        "reg_lambda": [1, 1.5, 2],
        "gamma": [0, 0.1, 0.4, 0.6],
        "min_child_weight": [1, 2, 4]}

    # creat estimator
    model = XGBRegressor(objective = 'reg:squarederror')

    # create the grid search object
    grid = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_grid,
        cv=5,
        scoring= 'neg_mean_absolute_error',
        n_jobs=-1,
        n_iter=n_iter)

    print('RandomSearchCV ...', day_i)
    grid.fit(aggregated_data_m.values, aggregated_label_m.values)
    results = pd.DataFrame(grid.cv_results_).sort_values('mean_test_score', ascending=False)
    print('RandomSearchCV Done!')

    model = XGBRegressor(objective = 'reg:squarederror', **results.params.iloc[0,])
    model.fit(aggregated_data_m.values, aggregated_label_m.values)
    importance_list.append(model.feature_importances_)

In [ ]:
importance_df = pd.DataFrame(importance_list, columns=aggregated_data_m.columns)
importance_df

In [ ]:
importance_df.to_csv('importance_df.csv', index=False)

## Visualize Feature Importance

In [ ]:
fig, ax = plt.subplots()

ax = sns.barplot(data=importance_df)
ax = plt.plot(list(importance_df.iloc[[len(importance_df)-1]].values[0]), marker='o', ls='', color='k', label='Last Round')

plt.legend()
plt.xticks(rotation=30)

Save and download from the directory

In [ ]:
fig.savefig('importance_df.png', format='png', dpi=1200)
fig.savefig('importance_df.svg', format='svg', dpi=1200)

# Find NonLinear Interactions

<p1><h2> When to use this part:</h2>
* you have at least one Results.csv file </p1>
<p1><h2> How to use this part:</h2>
* run all cells!
* you can download Interactions.png from Files</p1>

In [ ]:
day = day_finder('Results') - 1
start_day = (1, 0)[os.path.isfile('Results_0.csv')]

desired_cols = []
for key, value in concentrations_limits.items():
    if value['Alternatives']:
        desired_cols.append(key)
        alternative_name = ['{}_{}'.format(key, i) for i in value['Alternatives']]
        desired_cols += alternative_name
    else:
        desired_cols.append(key)

final_order = desired_cols

aggregated_data_m = pd.DataFrame(columns=desired_cols)
aggregated_label_m = pd.DataFrame(columns=['yield'])

for num in range(start_day, day + 1):
    if start_day or num>0:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, days_range[num-1])
    else:
        data_m, label_m, _, _= result_preprocess(num, desired_cols, len(pd.read_csv('Results_0.csv')))

    aggregated_data_m = pd.concat([aggregated_data_m, data_m]).reset_index(drop=True)
    aggregated_label_m = pd.concat([aggregated_label_m, label_m]).reset_index(drop=True)


aggregated_data_m

In [ ]:
from sklearn.linear_model import LinearRegression
from itertools import combinations

model = LinearRegression()
model.fit(aggregated_data_m.values, aggregated_label_m.values)
baseline_score = pearsonr(pd.DataFrame(model.predict(aggregated_data_m.values))[0] ,aggregated_label_m['yield'])
baseline_score

In [ ]:
interaction_scores = {}
for i in combinations(aggregated_data_m.columns, 2):
    X = pd.concat([aggregated_data_m, (aggregated_data_m[i[0]] * aggregated_data_m[i[1]])], axis=1)
    y = aggregated_label_m

    model = LinearRegression()
    model.fit(X.values, y.values)
    score = pearsonr(pd.DataFrame(model.predict(X.values))[0] ,y['yield'])

    interaction_scores[i] = score

In [ ]:
interactions_df = pd.DataFrame(data=np.zeros([len(aggregated_data_m.columns), len(aggregated_data_m.columns)]), columns=aggregated_data_m.columns, index=aggregated_data_m.columns)

for columns, value in interaction_scores.items():
    diff = value[0] - baseline_score[0]
    interactions_df[columns[0]][columns[1]] = diff

In [ ]:
corr = np.corrcoef(np.random.randn(len(aggregated_data_m.columns), 200))
mask = np.zeros_like(corr)
mask[np.triu_indices_from(mask)] = True

fig, ax = plt.subplots(figsize = (7,5))
ax = sns.heatmap(interactions_df, mask=mask, cmap='YlGnBu')

In [ ]:
fig.savefig('Interactions.png', format='png', dpi=1200)
fig.savefig('Interactions.svg', format='svg', dpi=1200)

In [ ]:
interactions_df.to_csv('Interactions.csv')